# Microscopy dataframe operations

Apply reproducible feature filtering to an analysis-ready microscopy table.

In [ ]:
import pandas as pd
from nuclear_table_tools import (
    AdvancedOperator, drop_high_corr_columns, drop_low_unique_columns, merge_with_VAE_df,
)


In [ ]:
table = pd.DataFrame({
    'condition': ['Day1'] * 4,
    'batch': ['B1'] * 4,
    'subbatch': ['S1'] * 4,
    'label-id': [1, 2, 3, 4],
    'file': ['/images/sample_a.nd2'] * 4,
    'area': [20.0, 24.0, 22.0, 28.0],
    'DAPI_mean': [1.0, 2.0, 3.0, 4.0],
    'DAPI_duplicate': [1.0, 2.0, 3.0, 4.0],
    'DAPI_constant': [1.0, 1.0, 1.0, 1.0],
})
feature_columns = [column for column in table if column.startswith('DAPI_')]
table = drop_low_unique_columns(table, feature_columns, unique_threshold=2)
feature_columns = [column for column in feature_columns if column in table]
table = drop_high_corr_columns(table, feature_columns, corr_threshold=0.95)
table


In [ ]:
latent_table = pd.DataFrame({
    'id': range(4),
    'label': ['Day1'] * 4,
    'batch_label': ['B1'] * 4,
    'subbatch_label': ['S1'] * 4,
    'nuc_label': [1, 2, 3, 4],
    'key': [f'/crops/sample_a_nuc_{label}.tif' for label in [1, 2, 3, 4]],
    'feature_0': [0.12, 0.34, 0.56, 0.78],
})
combined = merge_with_VAE_df(table, latent_table)
combined[['label-id', 'DAPI_mean', 'feature_0']]


In [ ]:
operator = AdvancedOperator(combined)
operator.add_column('DAPI_mean_to_area', combined['DAPI_mean'] / combined['area'])
operator.filter_rows(lambda frame: frame['DAPI_mean_to_area'].notna())
analysis_table = operator.get_dataframe()
analysis_table
